In [53]:
from read_model_runs import read_model_runs

question_long_df, model_wide_accuracy_df, question_wide_accuracy_df, firac_order, model_order = read_model_runs('../data/processed/model-runs')

# Filtrar apenas as linhas em português
question_long_df = question_long_df[question_long_df["language"] == "portuguese"].copy()

print('shape', question_long_df.shape)
print('# unique questions', question_long_df['question_id'].nunique())
print('firac_order', firac_order)
print('model_order', model_order)

question_long_df = question_long_df[
    question_long_df["model_name"].str.contains("gemma", case=False, na=False) &
    ~question_long_df["model_name"].str.contains("gemma-3n", case=False, na=False)
]

question_long_df = question_long_df[question_long_df["firac"].ne("F____")]
question_long_df = question_long_df[question_long_df["firac"].ne("unstructured")]


question_long_df.head(1)




shape (56780, 30)
# unique questions 2920
firac_order ['FILA_', 'FIR__', 'FI___', 'FIL__', 'F____', '_____', 'unstructured']
model_order ['gemini-2.0-flash-lite', 'gemini-2.5-flash-lite', 'gemma-3-27b-it', 'gemma-3-12b-it', 'gemma-3n-e4b-it', 'gemma-3n-e2b-it', 'gemma-3-4b-it']


,model_name,firac,language,is_correct,pdf_filename,question_id,materia,oab_test_id,oab_question_id,chosen_option,...,D,full_response,Facts,Issue,Rule,Application,Conclusion,rule_count,fact_count,response_time_seconds
1976,gemma-3-12b-it,FILA_,portuguese,True,oab-153.pdf,oab-153.pdf-007,DIREITO CIVIL,II,7,C,...,cada herdeiro pode ser demandado pela dívida t...,"{\n ""F"":[\n ""A existência de um regime de sol...",['A existência de um regime de solidariedade p...,Qual a correta aplicação das regras do regime ...,"Art. 276 do Código Civil, Art. 279 do Código C...",Ao analisar a responsabilidade em caso de fale...,NaN,4,5,10.862945


In [54]:

# accuracy, input_token_count, output_token_count, fact_count_ rule_count

import pandas as pd

question_counts_df = (
    question_long_df
        .groupby("question_id", as_index=False)
        .agg(
            accuracy=("is_correct", "mean"),
            input_token_count=("input_token_count", "first"),
          #  output_token_count=("output_token_count", "mean"),
            fact_count=("fact_count", "first"),
            rule_count=("rule_count", "first"),
        )
)

question_counts_df.head()

,question_id,accuracy,input_token_count,fact_count,rule_count
0,oab-1.pdf-002,0.866667,791,7,3
1,oab-1.pdf-003,0.533333,1096,6,4
2,oab-1.pdf-004,0.733333,587,4,1
3,oab-1.pdf-005,0.642857,703,3,2
4,oab-1.pdf-006,0.866667,670,6,1


In [55]:
import pandas as pd

# 1. Cria coluna temporária que garante que cada FIRAC esteja na mesma ordem da lista firac_order
question_long_df["firac"] = pd.Categorical(
    question_long_df["firac"],
    categories=firac_order,
    ordered=True
)

# 2. Pivot para ter 1 coluna por FIRAC e valores 0/1 de is_correct
pivot = (
    question_long_df
    .pivot_table(
        index=["model_name", "question_id"], 
        columns="firac",
        values="is_correct",
        aggfunc="max",       # caso haja duplicatas; pode trocar por first
        fill_value=0
    )
)

# 3. Ordena as colunas conforme firac_order
# Mantém apenas a interseção entre firac_order e pivot.columns
firac_cols = [col for col in firac_order if col in pivot.columns]
# Ordena o pivot apenas com as colunas existentes
pivot = pivot[firac_cols]


# 4. Gera a coluna final com o arranjo de 0/1
pivot["firac_vector"] = pivot.apply(lambda row: row.values.tolist(), axis=1)

# 5. Adiciona a coluna "consistent"
def is_consistent(vec):
    seen_zero = False
    for x in vec:
        if x == 0:
            seen_zero = True
        elif seen_zero and x == 1:
            return False
    return True

pivot["consistent"] = pivot["firac_vector"].apply(is_consistent)

# Resultado final
model_question_firac_seq_df = pivot.reset_index()[["model_name", "question_id", "firac_vector", "consistent"]]
model_question_firac_seq_df


C:\Users\pedro\AppData\Local\Temp\ipykernel_16132\4181883816.py:13: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  .pivot_table(


firac,model_name,question_id,firac_vector,consistent
0,gemma-3-12b-it,oab-1.pdf-002,"[True, True, True, True, False]",True
1,gemma-3-12b-it,oab-1.pdf-003,"[True, True, False, False, False]",True
2,gemma-3-12b-it,oab-1.pdf-004,"[True, True, True, True, False]",True
3,gemma-3-12b-it,oab-1.pdf-005,"[True, False, True, True, 0]",False
4,gemma-3-12b-it,oab-1.pdf-006,"[True, True, True, True, True]",True
...,...,...,...,...
8032,gemma-3-4b-it,oab-99.pdf-007,"[True, False, False, False, True]",False
8033,gemma-3-4b-it,oab-99.pdf-008,"[True, True, False, False, False]",True
8034,gemma-3-4b-it,oab-99.pdf-009,"[True, True, True, False, True]",False
8035,gemma-3-4b-it,oab-99.pdf-010,"[True, True, True, True, True]",True


In [56]:
import pandas as pd

# 1
# 7. 👉 NOVO: consistência média por questão
question_consistency_df = (
    model_question_firac_seq_df
    .groupby("question_id", as_index=False)
    .agg(avg_consistency=("consistent", "mean"))
)

question_consistency_df.describe()


,avg_consistency
count,2725.000000
mean,0.667034
std,0.336984
min,0.000000
25%,0.333333
50%,0.666667
75%,1.000000
max,1.000000


In [ ]:
import pandas as pd

nli_df = pd.read_csv("../data/processed/question_entropy_ground-truth_gemma-3-27b-it.csv")

nli_df.columns

Index(['question_id', 'materia', 'firac_step', 'model_1', 'model_1_is_correct',
       'model_2', 'model_2_is_correct', 'logic_relation_rule', 'model_1_rule',
       'model_2_rule', 'logic_relation_application', 'model_1_application',
       'model_2_application', 'logic_relation_conclusion',
       'model_1_conclusion', 'model_2_conclusion'],
      dtype='object')

In [8]:
import pandas as pd

logic_cols = [
    "logic_relation_application",
    "logic_relation_rule",
    "logic_relation_conclusion"
]

dfs = []

# Primeiro: gerar contagens por question_id para cada tipo
for col in logic_cols:
    df_col = (
        nli_df
        .groupby(["question_id", col])
        .size()
        .unstack(fill_value=0)
    )

    df_col.columns = [f"{col}_{v}" for v in df_col.columns]
    dfs.append(df_col)

# Unir tudo
nli_stats_df = pd.concat(dfs, axis=1).fillna(0)

# ---------------------------------------------------
# Agora: criar logic_relation_total_[valor]
# ---------------------------------------------------
# Descobrir todos os valores possíveis
values = set(
    v.replace("logic_relation_application_", "")
     .replace("logic_relation_rule_", "")
     .replace("logic_relation_conclusion_", "")
    for v in nli_stats_df.columns
    if v.startswith("logic_relation_")
)

for v in values:
    cols_v = [c for c in nli_stats_df.columns if c.endswith(f"_{v}")]
    nli_stats_df[f"logic_relation_total_{v}"] = nli_stats_df[cols_v].sum(axis=1)

nli_stats_df = nli_stats_df.reset_index()

nli_stats_df.head()

,question_id,logic_relation_application_CONTRADICTION,logic_relation_application_ENTAILMENT,logic_relation_application_NEUTRAL,logic_relation_rule_CONTRADICTION,logic_relation_rule_ENTAILMENT,logic_relation_rule_NEUTRAL,logic_relation_conclusion_CONTRADICTION,logic_relation_conclusion_ENTAILMENT,logic_relation_conclusion_NEUTRAL,logic_relation_total_NEUTRAL,logic_relation_total_ENTAILMENT,logic_relation_total_CONTRADICTION
0,oab-1.pdf-002,0,1,0,0,1,0,0,1,0,0,3,0
1,oab-1.pdf-003,0,1,0,0,1,0,0,1,0,0,3,0
2,oab-1.pdf-004,0,1,0,0,1,0,0,1,0,0,3,0
3,oab-1.pdf-005,0,1,0,1,0,0,0,1,0,0,2,1
4,oab-1.pdf-006,0,1,0,0,1,0,0,1,0,0,3,0


In [57]:
question_agg_df = question_counts_df.merge(
    question_consistency_df,
    on="question_id",
    how="left"
)

question_agg_df = question_agg_df.reset_index()
question_agg_df = question_agg_df.drop(columns=["index"])


question_agg_df.head()

,question_id,accuracy,input_token_count,fact_count,rule_count,avg_consistency
0,oab-1.pdf-002,0.866667,791,7,3,1.000000
1,oab-1.pdf-003,0.533333,1096,6,4,1.000000
2,oab-1.pdf-004,0.733333,587,4,1,1.000000
3,oab-1.pdf-005,0.642857,703,3,2,0.000000
4,oab-1.pdf-006,0.866667,670,6,1,0.666667


In [58]:
question_agg_df.columns

Index(['question_id', 'accuracy', 'input_token_count', 'fact_count',
       'rule_count', 'avg_consistency'],
      dtype='object')

In [59]:
question_agg_df.to_csv("../data/processed/question_statistics.csv", index=False)